In [1]:
import pandas as pd
import numpy as np
#将数据输出结果导出
import os

In [2]:
df=pd.read_csv('out_data/allmonth_clean.csv')
# 字段合法性校验
required_cols = ['user_id', 'product_id', 'category_id', 'event_type', 'event_time', 'hour', 'date']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    print(f'⚠️ 警告：缺少必要字段 {missing_cols}，请检查数据')
else:
    print('✅ 字段校验通过')
print('-'*55)
'''---------1.时段PV/UV:按小时维度统计浏览量、独立用户数---------'''
hourly_pv_uv = df[df['event_type']=='view'].groupby('hour').agg(
    pv=('user_id', 'count'),
    uv=('user_id', 'nunique')
).reset_index()
print(f"1. 时段PV/UV统计:{hourly_pv_uv.to_string(index=False)}")
print('-'*55)
#定位用户活跃高峰时段，指导活动上线、推送投放时间
#人均浏览量:平均每个用户浏览的页面数 = PV / UV
daily_metric = df[df['event_type']=='view'].groupby('date').agg(
    pv=('user_id', 'count'),
    uv=('user_id', 'nunique')
).reset_index()
daily_metric['avg_page_view'] = daily_metric['pv'] / daily_metric['uv']
print(f'2. 每日流量和人均浏览量:{daily_metric.to_string(index=False)}')
print('-'*55)
'''---------2.转化指标:均为用户维度转化率:独立用户去重计算,并不是用次数占比
,加购率：浏览用户中产生加购行为的用户占比（标准业务口径：分母=浏览用户）
---------'''
'''直接通过行为筛选+去重计算，替代groupby+lambda写法，性能更优
user_cart = df.groupby('user_id')['event_type'].agg(
lambda x: (x == 'cart').any()).reset_index(name='has_cart')'''
view_user_ids = df[df['event_type'] == 'view']['user_id'].unique()
total_uv = len(view_user_ids)
# 分子：有加购行为的独立用户
cart_user = df[(df['event_type'] == 'cart') & df['user_id'].isin(view_user_ids)]['user_id'].nunique()
cart_rate = cart_user / total_uv
print("3. 用户维度加购率")
print(f"浏览独立用户数（分母）：{total_uv}")
print(f"有加购行为用户数（分子）：{cart_user}")
print(f"用户加购率：{cart_rate:.4f}（{cart_rate*100:.2f}%）")
print("-"*60)
# 购买转化率：浏览用户中最终下单购买的用户占比
# 与加购率统一口径，分母复用浏览用户数，保证指标横向可比
purchase_user_ids = df[df['event_type'] == 'purchase']['user_id'].unique()
purchase_user = len(set(purchase_user_ids) & set(view_user_ids))

purchase_rate = purchase_user / total_uv
print("4. 用户维度购买转化率")
print(f"浏览独立用户数（分母）：{total_uv}")
print(f"有购买行为用户数（分子）：{purchase_user}")
print(f"用户购买转化率：{purchase_rate:.4f}（{purchase_rate*100:.2f}%）")
print("-"*60)
# 分品类购买转化率：各品类下，浏览过该品类的用户的购买转化率
# 逻辑：分别统计各品类的浏览用户数（分母）、购买用户数（分子），再合并计算转化率
# 1. 分母：每个品类的浏览用户数
cat_view_uv = df[df['event_type'] == 'view'].groupby('category_id')['user_id'].nunique().reset_index(name='cat_view_uv')

# 2. 分子：每个品类的购买用户数
cat_buy_uv = df[df['event_type'] == 'purchase'].groupby('category_id')['user_id'].nunique().reset_index(name='cat_buy_uv')

# 3. 合并计算转化率
cat_convert = pd.merge(cat_view_uv, cat_buy_uv, on='category_id', how='left')
cat_convert['convert_rate'] = cat_convert['cat_buy_uv'] / cat_convert['cat_view_uv']
cat_convert['convert_rate'] = cat_convert['convert_rate'].fillna(0)  # 处理无转化的品类
cat_convert = cat_convert.sort_values('convert_rate', ascending=False).reset_index(drop=True)
#区分零转化的品类
cat_convert_has_buy = cat_convert[cat_convert['convert_rate'] > 0]

print("5. 分品类购买转化率（按浏览用户口径）")
print(cat_convert_has_buy.to_string(index=False))
print("-"*60)
#零转化的品类
zero_convert_count = (cat_convert['convert_rate'] == 0).sum()
total_cat_count = len(cat_convert)
print(f"零转化品类数：{zero_convert_count} 个，占有浏览品类总数的 {zero_convert_count/total_cat_count*100:.2f}%")
'''---------3. 商品维度指标
核心指标：热销品类TOP10、销量TOP10商品、点击量TOP10商品
---------'''
'''销量TOP10商品（双维度统计）
# 维度1：按订单数统计 → 反映商品整体销量规模
# 维度2：按购买人数统计 → 反映商品受众覆盖广度'''
# 1：按购买订单数统计（原逻辑，保留）
top10_item_sale_cnt = df[df['event_type'] == 'purchase'].groupby('product_id').size()\
    .reset_index(name='sale_count').sort_values('sale_count', ascending=False).head(10)

# 2：按购买人数统计（新增，业务价值更高）
top10_item_sale_usr = df[df['event_type'] == 'purchase'].groupby('product_id')['user_id'].nunique()\
    .reset_index(name='buy_user_count').sort_values('buy_user_count', ascending=False).head(10)
print("销量TOP10商品（按订单数)")
print(top10_item_sale_cnt.to_string(index=False))
print(" 销量TOP10商品（按购买人数）")
print(top10_item_sale_usr.to_string(index=False))
# 点击量TOP10商品
top10_item_click = df[df['event_type'] == 'view'] .groupby('product_id').size() .reset_index(name='click_count') .sort_values('click_count', ascending=False) .head(10)
print("浏览量TOP10商品")
print(top10_item_click.to_string(index=False))
# 热销品类TOP10
top10_category = df[df['event_type'] == 'purchase'] .groupby('category_id').size() .reset_index(name='sale_count') .sort_values('sale_count', ascending=False) .head(10)
print("销量TOP10品类")
print(top10_category.to_string(index=False))
#======================保存输出结果用于可视化输出======================
daily_metric.to_csv('output/data/daily_pv_uv.csv',
                    index=False, encoding='utf-8-sig')

# 时段流量
hourly_pv_uv.to_csv('output/data/hour_pv.csv',
                    index=False, encoding='utf-8-sig')

#2. 转化模块
# 分品类转化率
cat_convert.to_csv('output/data/category_conv.csv',
                   index=False, encoding='utf-8-sig')

# 构造整体转化漏斗表（把分散的数值拼成表）
df_funnel = pd.DataFrame({
    'stage': ['浏览', '加购', '购买'],
    'user_count': [total_uv, cart_user, purchase_user]
})
df_funnel.to_csv('output/data/conversion_funnel.csv',
                 index=False, encoding='utf-8-sig')

# 3. 商品模块
# 商品销量TOP10-按订单数
top10_item_sale_cnt.to_csv('output/data/product_sales_cnt.csv',
                           index=False, encoding='utf-8-sig')

# 商品销量TOP10-按购买人数
top10_item_sale_usr.to_csv('output/data/product_sales_usr.csv',
                           index=False, encoding='utf-8-sig')

# 商品浏览TOP10
top10_item_click.to_csv('output/data/product_click.csv',
                        index=False, encoding='utf-8-sig')

# 品类销量TOP10
top10_category.to_csv('output/data/category_sales.csv',
                      index=False, encoding='utf-8-sig')

print(" 所有结果已保存到 output/data/ 目录")


✅ 字段校验通过
-------------------------------------------------------
1. 时段PV/UV统计: hour     pv     uv
    0  89566  26677
    1  76742  25454
    2  84697  29906
    3 121696  42543
    4 181272  61588
    5 273060  90251
    6 375305 118357
    7 452895 140769
    8 500858 154083
    9 526941 159890
   10 555913 165485
   11 572513 168519
   12 580717 171343
   13 556927 163460
   14 522165 156071
   15 500161 151470
   16 514585 153708
   17 559153 160177
   18 604851 164839
   19 639029 162941
   20 572947 137943
   21 387829  92556
   22 232863  56786
   23 135557  35452
-------------------------------------------------------
2. 每日流量和人均浏览量:      date     pv    uv  avg_page_view
2019-10-01  61042 18378       3.321471
2019-10-02  76316 32939       2.316889
2019-10-03  56627 15445       3.666365
2019-10-04  53511 13921       3.843905
2019-10-05  49312 14313       3.445260
2019-10-06  72763 30627       2.375780
2019-10-07  75687 25408       2.978865
2019-10-08  64788 19198       3.374727
2